In [2]:
import zipfile
import os


DATASET_PATH = "/content/RPC"
os.makedirs(DATASET_PATH, exist_ok=True)


zip_file_path = "/content/archive.zip"

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(DATASET_PATH)

print(f"Extracted archive to {DATASET_PATH}")

Extracted archive to /content/RPC


In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.3 MB/s eta 0:00:00


In [7]:
import glob

label_files = glob.glob(os.path.join(train_output_labels_dir, '*.txt'))

if label_files:
    sample_label_file = label_files[0]
    print(f"\n--- Inspecting sample label file: {sample_label_file} ---")
    with open(sample_label_file, 'r') as f:
        content = f.read()
        print(content)
    print(f"--- End of sample label file: {sample_label_file} ---")
else:
    print(f"No label files found in {train_output_labels_dir}")


--- Inspecting sample label file: /content/RPC/dataset/labels/train/20210518_202038.txt ---
5 0.440430 0.479492 0.494141 0.474609

--- End of sample label file: /content/RPC/dataset/labels/train/20210518_202038.txt ---


In [8]:
import os
import xml.etree.ElementTree as ET
from ultralytics import YOLO

BASE_EXTRACTION_PATH = "/content/RPC"

DATASET_ROOT_PATH = os.path.join(BASE_EXTRACTION_PATH, "dataset")

COCO_ANNOTATIONS_DIR = os.path.join(DATASET_ROOT_PATH, "annotations")

os.makedirs(DATASET_ROOT_PATH, exist_ok=True)

def convert_pascal_voc_to_yolo(xml_dir, image_dir, output_dir, classes):
    """
    Converts Pascal VOC XML annotations to YOLO format TXT files.
    Args:
        xml_dir (str): Directory containing Pascal VOC XML annotation files.
        image_dir (str): Directory containing corresponding image files (to get image dimensions).
        output_dir (str): Directory to save YOLO format TXT label files.
        classes (list): List of class names in the desired order (index will be class ID).
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nStarting conversion in {xml_dir} to {output_dir}")

    for xml_file in os.listdir(xml_dir):
        if not xml_file.endswith('.xml'):
            continue

        xml_path = os.path.join(xml_dir, xml_file)
        print(f"Processing XML file: {xml_file}")
        tree = ET.parse(xml_path)
        root = tree.getroot()

        img_name_without_ext = os.path.splitext(xml_file)[0]
        inferred_image_filename = img_name_without_ext + '.jpg'
        print(f"  Inferred image filename for label generation: {inferred_image_filename}")

        size = root.find('size')
        if size is None:
            print(f"Error: <size> tag not found in {xml_file}. Skipping.")
            continue
        try:
            img_width = int(size.find('width').text)
            img_height = int(size.find('height').text)
            print(f"  Image dimensions from XML for {inferred_image_filename}: {img_width}x{img_height}")
        except (ValueError, AttributeError) as e:
            print(f"Error parsing image dimensions in {xml_file}: {e}. Skipping.")
            continue

        yolo_labels = []
        objects_found = 0

        for obj in root.findall('object'):
            objects_found += 1
            class_name = obj.find('name').text.strip()
            print(f"    Found object class: '{class_name}'")

            if class_name not in classes:
                print(f"      Warning: Class '{class_name}' not found in defined classes {classes}. Skipping object.")
                continue
            class_id = classes.index(class_name)

            bndbox = obj.find('bndbox')
            if bndbox is None:
                print(f"      Error: <bndbox> tag not found for object '{class_name}' in {xml_file}. Skipping object.")
                continue
            try:
                xmin = int(bndbox.find('xmin').text)
                ymin = int(bndbox.find('ymin').text)
                xmax = int(bndbox.find('xmax').text)
                ymax = int(bndbox.find('ymax').text)
                print(f"      Bbox coords: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")
            except (ValueError, AttributeError) as e:
                print(f"      Error parsing bbox coordinates for object '{class_name}' in {xml_file}: {e}. Skipping object.")
                continue

            x_center = (xmin + xmax) / 2 / img_width
            y_center = (ymin + ymax) / 2 / img_height
            box_width = (xmax - xmin) / img_width
            box_height = (ymax - ymin) / img_height

            if img_width == 0 or img_height == 0:
                print(f"      Error: Image width or height is zero, cannot normalize bounding box for {xml_file}. Skipping object.")
                continue

            yolo_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}")
            print(f"      Generated YOLO label: {yolo_labels[-1]}")

        print(f"  Total objects found in {xml_file}: {objects_found}")
        print(f"  Total YOLO labels generated for {xml_file}: {len(yolo_labels)}")

        if yolo_labels:
            output_label_path = os.path.join(output_dir, f"{img_name_without_ext}.txt")
            with open(output_label_path, 'w') as f:
                for line in yolo_labels:
                    f.write(line + '\n')
            print(f"  Successfully wrote {len(yolo_labels)} labels to {output_label_path}")
        else:
            print(f"  No YOLO labels to write for {xml_file}.")


def get_unique_classes_from_xml(xml_dirs):
    """
    Extracts all unique class names from Pascal VOC XML files across multiple directories.
    Args:
        xml_dirs (list): A list of directories containing Pascal VOC XML annotation files.
    Returns:
        list: A sorted list of unique class names.
    """
    unique_classes = set()
    for xml_dir in xml_dirs:
        if not os.path.exists(xml_dir):
            print(f"Warning: XML directory not found: {xml_dir}")
            continue
        for xml_file in os.listdir(xml_dir):
            if xml_file.endswith('.xml'):
                xml_path = os.path.join(xml_dir, xml_file)
                try:
                    tree = ET.parse(xml_path)
                    root = tree.getroot()
                    for obj in root.findall('object'):
                        class_name = obj.find('name').text.strip()
                        unique_classes.add(class_name)
                except ET.ParseError:
                    print(f"Error parsing XML file: {xml_path}")
    return sorted(list(unique_classes))


train_xml_dir = os.path.join(COCO_ANNOTATIONS_DIR, "train")
train_image_dir = os.path.join(DATASET_ROOT_PATH, "images", "train")
train_output_labels_dir = os.path.join(DATASET_ROOT_PATH, "labels", "train")

test_xml_dir = os.path.join(COCO_ANNOTATIONS_DIR, "test")
test_image_dir = os.path.join(DATASET_ROOT_PATH, "images", "test")
test_output_labels_dir = os.path.join(DATASET_ROOT_PATH, "labels", "test")

CLASS_NAMES = get_unique_classes_from_xml([train_xml_dir, test_xml_dir])

if not CLASS_NAMES:
    raise ValueError("No classes found in XML annotation files. Please check the dataset.")

print(f"Discovered classes: {CLASS_NAMES}")

print("Converting training annotations...")
convert_pascal_voc_to_yolo(train_xml_dir, train_image_dir, train_output_labels_dir, CLASS_NAMES)

print("Converting validation annotations...")
convert_pascal_voc_to_yolo(test_xml_dir, test_image_dir, test_output_labels_dir, CLASS_NAMES)

print("Pascal VOC to YOLO conversion complete.")

yaml_content = f"""
path: {DATASET_ROOT_PATH} # This path is the root of the dataset containing 'images' and 'labels' directories

train: images/train # Images are expected to be in 'images/train' relative to path
val: images/test    # Images are expected to be in 'images/test' relative to path (using test for validation)

# Dynamically generated names for classes
names:
"""
for i, name in enumerate(CLASS_NAMES):
    yaml_content += f"  {i}: {name}\n"

with open(os.path.join(DATASET_ROOT_PATH, "data.yaml"), "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully")

print(f"\nListing contents of {DATASET_ROOT_PATH}:")
!ls -R {DATASET_ROOT_PATH}


model = YOLO("yolov8s.pt")

# Train
model.train(
    data=os.path.join(DATASET_ROOT_PATH, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    project="rpc_yolo",
    name="product_detector"
)

# Validate
metrics = model.val()
print(metrics)

# Export model
model.export(format="onnx")

print("Training Completed Successfully")


Streaming output truncated to the last 5000 lines.
  Successfully wrote 6 labels to /content/RPC/dataset/labels/train/mix (29).txt
Processing XML file: mix (23).xml
  Inferred image filename for label generation: mix (23).jpg
  Image dimensions from XML for mix (23).jpg: 512x512
    Found object class: 'pepsodent'
      Bbox coords: xmin=56, ymin=258, xmax=218, ymax=314
      Generated YOLO label: 3 0.267578 0.558594 0.316406 0.109375
    Found object class: 'shampoo'
      Bbox coords: xmin=175, ymin=198, xmax=259, ymax=347
      Generated YOLO label: 4 0.423828 0.532227 0.164062 0.291016
    Found object class: 'tissue'
      Bbox coords: xmin=221, ymin=215, xmax=384, ymax=305
      Generated YOLO label: 5 0.590820 0.507812 0.318359 0.175781
    Found object class: 'aqua'
      Bbox coords: xmin=379, ymin=139, xmax=495, ymax=363
      Generated YOLO label: 0 0.853516 0.490234 0.226562 0.437500
    Found object class: 'chitato'
      Bbox coords: xmin=87, ymin=278, xmax=230, ymax=432
